In [ ]:
from __future__ import annotations
import os
import subprocess
import sys
import time
from pathlib import Path
from dotenv import load_dotenv


def resolve_project_root_from_cwd(cwd: Path) -> Path:
    if (cwd / "main.ipynb").exists() and (cwd / "standalone_scripts").exists():
        return cwd
    if cwd.name == "standalone_scripts":
        parent = cwd.parent
        if (parent / "main.ipynb").exists():
            return parent
        if (cwd / "main_extractor.py").exists() and (cwd / "date_expander.py").exists():
            return parent
    scripts = cwd / "standalone_scripts"
    if scripts.is_dir() and (scripts / "main_extractor.py").exists():
        return cwd
    raise RuntimeError(
        f"Unable to determine project root from cwd={cwd}. "
        "Open the repo root or standalone_scripts/ as the notebook folder, or restore main.ipynb."
    )


def parse_starting_page_urls(raw: str | None) -> list[str]:
    if raw is None or not str(raw).strip():
        raise RuntimeError(
            "ACTIVE_STARTING_PAGE_URLS is not set or empty. "
            "Set it to one or more URLs separated by commas (or newlines)."
        )
    urls: list[str] = []
    for chunk in raw.replace("\n", ",").split(","):
        url = chunk.strip()
        if url:
            urls.append(url)
    if not urls:
        raise RuntimeError(
            "ACTIVE_STARTING_PAGE_URLS contained no non-empty URLs after parsing."
        )
    return urls


print("Starting orchestrator")
CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = resolve_project_root_from_cwd(CURRENT_WORKING_DIRECTORY)
_ = load_dotenv(PROJECT_ROOT / ".env")

SCRIPTS_DIR = PROJECT_ROOT / "standalone_scripts"
MAIN_EXTRACTOR_PATH = SCRIPTS_DIR / "main_extractor.py"
DATE_EXPANDER_PATH = SCRIPTS_DIR / "date_expander.py"

VENV_PYTHON_PATH = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON_RUNNER = str(VENV_PYTHON_PATH if VENV_PYTHON_PATH.exists() else Path(sys.executable))

STARTING_PAGE_URLS = parse_starting_page_urls(os.environ.get("ACTIVE_STARTING_PAGE_URLS"))
DATE_EXPANDER_LIMIT = 500


def run_command(command_parts: list[str], label: str) -> int:
    print(f"\n[{label}] Running command:")
    print(" ".join(command_parts))

    started_at = time.perf_counter()
    completed_process = subprocess.run(
        command_parts,
        cwd=str(PROJECT_ROOT),
        check=False,
    )
    elapsed_seconds = time.perf_counter() - started_at
    print(
        f"[{label}] Exit code={completed_process.returncode} elapsed={elapsed_seconds:.1f}s"
    )
    return completed_process.returncode


if not MAIN_EXTRACTOR_PATH.exists():
    raise FileNotFoundError(f"Missing script: {MAIN_EXTRACTOR_PATH}")
if not DATE_EXPANDER_PATH.exists():
    raise FileNotFoundError(f"Missing script: {DATE_EXPANDER_PATH}")

print(f"[orchestrator] Active URLs: {len(STARTING_PAGE_URLS)}")

url_results: list[dict[str, object]] = []
for url_index, starting_page_url in enumerate(STARTING_PAGE_URLS, start=1):
    url_label = f"URL {url_index}/{len(STARTING_PAGE_URLS)}"
    print(f"\n[orchestrator] {url_label} -> {starting_page_url}")

    command_parts = [
        PYTHON_RUNNER,
        str(MAIN_EXTRACTOR_PATH),
        "--starting-page-url",
        starting_page_url,
    ]
    return_code = run_command(command_parts=command_parts, label=url_label)

    url_results.append(
        {
            "url": starting_page_url,
            "return_code": return_code,
            "status": "ok" if return_code == 0 else "failed",
        }
    )

print("\n[orchestrator] URL pipeline summary")
for result in url_results:
    print(f"- {result['status']}: {result['url']} (code={result['return_code']})")

failed_count = sum(1 for result in url_results if result["return_code"] != 0)
print(
    f"[orchestrator] URL run complete. total={len(url_results)} failed={failed_count}"
)

print("\n[orchestrator] Running date expansion apply step")
date_expander_command_parts = [
    PYTHON_RUNNER,
    str(DATE_EXPANDER_PATH),
    "--mode",
    "apply",
    "--limit",
    str(DATE_EXPANDER_LIMIT),
]
date_expander_return_code = run_command(
    command_parts=date_expander_command_parts,
    label="date_expander",
)

if date_expander_return_code != 0:
    raise RuntimeError(
        f"date_expander.py failed with return code {date_expander_return_code}"
    )

print("\n[orchestrator] Done")